In [0]:
# Criando o schema da Silver Layer

spark.sql("""
CREATE SCHEMA IF NOT EXISTS silver
""")

DataFrame[]

In [0]:
from pyspark.sql.functions import (
    col, upper, trim, when, current_timestamp,
    round, cast, to_date, year, month
)

# 1. Silver - CO2 Emissions
df_co2_silver = (
    spark.table("bronze.co2_emissions_raw")
    .withColumn("country", upper(trim(col("country"))))
    .withColumn("year", col("year").cast("int"))
    .withColumn("co2_emissions", col("co2_emissions_mt").cast("double"))
    .withColumn("silver_timestamp", current_timestamp())
    .select("country", "year", "co2_emissions", "ingestion_timestamp", "silver_timestamp")
    .dropDuplicates()
)

df_co2_silver.write.format("delta").mode("overwrite").saveAsTable("silver.co2_emissions")
print(f"✅ Silver - CO2 Emissions: {df_co2_silver.count()} registros")

✅ Silver - CO2 Emissions: 1350 registros


In [0]:
# 2. Silver - Energy Mix
df_energy_silver = (
    spark.table("bronze.energy_mix_raw")
    .withColumn("country", upper(trim(col("country"))))
    .withColumn("year", col("year").cast("int"))
    .withColumn("renewable_pct", col("renewables_total_pct").cast("double"))
    .withColumn("fossil_pct", col("fossil_total_pct").cast("double"))
    .withColumn("nuclear_pct", col("nuclear_pct").cast("double"))
    .withColumn("silver_timestamp", current_timestamp())
    .select("country", "year", "renewable_pct", "fossil_pct", "nuclear_pct", "ingestion_timestamp", "silver_timestamp")
    .dropDuplicates()
)

df_energy_silver.write.format("delta").mode("overwrite").saveAsTable("silver.energy_mix")
print(f"✅ Silver - Energy Mix: {df_energy_silver.count()} registros")

✅ Silver - Energy Mix: 1350 registros


In [0]:
# 3. Silver - Temperature Anomaly
df_temp_silver = (
    spark.table("bronze.temperature_anomaly_raw")
    .withColumn("region", upper(trim(col("region"))))
    .withColumn("year_month", col("year_month"))
    .withColumn("temp_anomaly", col("temp_anomaly_c").cast("double"))
    .withColumn("silver_timestamp", current_timestamp())
    .select("region", "year_month", "temp_anomaly", "ingestion_timestamp", "silver_timestamp")
    .dropDuplicates()
)

df_temp_silver.write.format("delta").mode("overwrite").saveAsTable("silver.temperature_anomaly")
print(f"✅ Silver - Temperature Anomaly: {df_temp_silver.count()} registros")

✅ Silver - Temperature Anomaly: 2528 registros


In [0]:
# 4 .Silver - Climate Events

df_events_silver = (
    spark.table("bronze.climate_events_raw")
    .withColumn("region", upper(trim(col("region"))))
    .withColumn("event_date", col("date"))
    .withColumn("year", year(col("event_date")))
    .withColumn("silver_timestamp", current_timestamp())
    .dropDuplicates()
)

df_events_silver.write.format("delta").mode("overwrite").saveAsTable("silver.climate_events")
print(f"✅ Silver - Climate Events: {df_events_silver.count()} registros")

✅ Silver - Climate Events: 50 registros


In [0]:
# 5. Silver - Carbon Prices
df_carbon_silver = (
    spark.table("bronze.carbon_prices_raw")
    .withColumn("date", col("Date").cast("date"))
    .withColumn("price", col("Price").cast("double"))
    .withColumn("silver_timestamp", current_timestamp())
    .select("date", "price", "ingestion_timestamp", "silver_timestamp")
    .dropDuplicates()
)

df_carbon_silver.write.format("delta").mode("overwrite").saveAsTable("silver.carbon_prices")
print(f"✅ Silver - Carbon Prices: {df_carbon_silver.count()} registros")

✅ Silver - Carbon Prices: 15864 registros


In [0]:
# Salvando os dados tratados na Silver Layer
# Utilizando o formato delta. 

df_co2_silver.write.format("delta").mode("overwrite").saveAsTable("silver.co2_emissions")
df_energy_silver.write.format("delta").mode("overwrite").saveAsTable("silver.energy_mix")
df_temp_silver.write.format("delta").mode("overwrite").saveAsTable("silver.temperature_anomaly")
df_carbon_silver.write.format("delta").mode("overwrite").saveAsTable("silver.carbon_prices")
df_events_silver.write.format("delta").mode("overwrite").saveAsTable("silver.climate_events")

In [0]:
display(
    spark.sql("""
    SELECT *
    FROM silver.co2_emissions
    LIMIT 20
              """)

)

country,year,co2_emissions,ingestion_timestamp,silver_timestamp
UNITED STATES,2012,5329.7,2026-06-05T18:27:20.733Z,2026-06-05T18:46:24.265Z
INDIA,2013,2093.4,2026-06-05T18:27:20.733Z,2026-06-05T18:46:24.265Z
INDIA,2025,3015.3,2026-06-05T18:27:20.733Z,2026-06-05T18:46:24.265Z
RUSSIA,2018,1642.1,2026-06-05T18:27:20.733Z,2026-06-05T18:46:24.265Z
RUSSIA,2025,1884.6,2026-06-05T18:27:20.733Z,2026-06-05T18:46:24.265Z
RUSSIA,2026,1834.9,2026-06-05T18:27:20.733Z,2026-06-05T18:46:24.265Z
JAPAN,2007,1238.3,2026-06-05T18:27:20.733Z,2026-06-05T18:46:24.265Z
JAPAN,2026,931.5,2026-06-05T18:27:20.733Z,2026-06-05T18:46:24.265Z
GERMANY,2005,820.9,2026-06-05T18:27:20.733Z,2026-06-05T18:46:24.265Z
GERMANY,2016,714.1,2026-06-05T18:27:20.733Z,2026-06-05T18:46:24.265Z


In [0]:
display(
    spark.sql("""
    SELECT *
    FROM silver.energy_mix
    LIMIT 20
              """)

)

country,year,renewable_pct,fossil_pct,nuclear_pct,ingestion_timestamp,silver_timestamp
CHINA,2000,6.37,92.75,0.87,2026-06-05T18:27:29.600Z,2026-06-05T18:46:26.109Z
CHINA,2015,11.37,88.1,0.53,2026-06-05T18:27:29.600Z,2026-06-05T18:46:26.109Z
UNITED STATES,2005,5.97,85.51,8.52,2026-06-05T18:27:29.600Z,2026-06-05T18:46:26.109Z
UNITED STATES,2017,11.52,80.6,7.88,2026-06-05T18:27:29.600Z,2026-06-05T18:46:26.109Z
INDIA,2018,7.82,91.09,1.09,2026-06-05T18:27:29.600Z,2026-06-05T18:46:26.109Z
JAPAN,2004,5.74,79.84,14.42,2026-06-05T18:27:29.600Z,2026-06-05T18:46:26.109Z
JAPAN,2019,11.41,85.54,3.06,2026-06-05T18:27:29.600Z,2026-06-05T18:46:26.109Z
GERMANY,2002,3.31,84.27,12.42,2026-06-05T18:27:29.600Z,2026-06-05T18:46:26.109Z
GERMANY,2012,11.71,78.59,9.71,2026-06-05T18:27:29.600Z,2026-06-05T18:46:26.109Z
IRAN,2011,3.92,95.46,0.62,2026-06-05T18:27:29.600Z,2026-06-05T18:46:26.109Z


In [0]:
display(
    spark.sql("""
    SELECT *
    FROM silver.temperature_anomaly
    LIMIT 20
              """)

)

region,year_month,temp_anomaly,ingestion_timestamp,silver_timestamp
EUROPE,2000-03-01,0.733,2026-06-05T18:27:33.731Z,2026-06-05T18:46:28.301Z
NORTHERN_HEMISPHERE,2000-05-01,0.627,2026-06-05T18:27:33.731Z,2026-06-05T18:46:28.301Z
SOUTHERN_HEMISPHERE,2000-11-01,0.377,2026-06-05T18:27:33.731Z,2026-06-05T18:46:28.301Z
NORTH_AMERICA,2001-03-01,0.712,2026-06-05T18:27:33.731Z,2026-06-05T18:46:28.301Z
GLOBAL,2001-04-01,0.49,2026-06-05T18:27:33.731Z,2026-06-05T18:46:28.301Z
NORTHERN_HEMISPHERE,2001-11-01,0.7,2026-06-05T18:27:33.731Z,2026-06-05T18:46:28.301Z
GLOBAL,2001-12-01,0.548,2026-06-05T18:27:33.731Z,2026-06-05T18:46:28.301Z
TROPICS,2002-01-01,0.342,2026-06-05T18:27:33.731Z,2026-06-05T18:46:28.301Z
SOUTHERN_HEMISPHERE,2002-02-01,0.427,2026-06-05T18:27:33.731Z,2026-06-05T18:46:28.301Z
EUROPE,2002-03-01,0.898,2026-06-05T18:27:33.731Z,2026-06-05T18:46:28.301Z


In [0]:
display(
    spark.sql("""
    SELECT *
    FROM silver.carbon_prices
    LIMIT 20
              """)

)

date,price,ingestion_timestamp,silver_timestamp
2005-04-25,7.13,2026-06-05T18:27:42.569Z,2026-06-05T18:46:30.331Z
2005-05-09,8.15,2026-06-05T18:27:42.569Z,2026-06-05T18:46:30.331Z
2005-05-10,8.24,2026-06-05T18:27:42.569Z,2026-06-05T18:46:30.331Z
2005-06-17,10.56,2026-06-05T18:27:42.569Z,2026-06-05T18:46:30.331Z
2005-07-21,12.57,2026-06-05T18:27:42.569Z,2026-06-05T18:46:30.331Z
2005-08-23,15.61,2026-06-05T18:27:42.569Z,2026-06-05T18:46:30.331Z
2005-09-02,15.99,2026-06-05T18:27:42.569Z,2026-06-05T18:46:30.331Z
2005-09-23,16.91,2026-06-05T18:27:42.569Z,2026-06-05T18:46:30.331Z
2005-10-07,18.01,2026-06-05T18:27:42.569Z,2026-06-05T18:46:30.331Z
2005-10-10,17.92,2026-06-05T18:27:42.569Z,2026-06-05T18:46:30.331Z


In [0]:
display(
    spark.sql("""
    SELECT *
    FROM silver.climate_events
    LIMIT 20
              """)

)

event_id,date,year,month,region,event_type,severity_score,description,is_policy,is_extreme_weather,is_disaster,ingestion_timestamp,event_date,silver_timestamp
CE011,2015-04-25,2015,4,NEPAL,earthquake,8,Nepal earthquake,0,0,1,2026-06-05T18:27:37.561Z,2015-04-25,2026-06-05T18:46:32.255Z
CE039,2024-04-15,2024,4,UAE,flood,7,Dubai flooding from extreme rainfall,0,1,0,2026-06-05T18:27:37.561Z,2024-04-15,2026-06-05T18:46:32.255Z
CE046,2025-06-20,2025,6,EUROPE,heatwave,7,European heatwave 45°C+,0,1,0,2026-06-05T18:27:37.561Z,2025-06-20,2026-06-05T18:46:32.255Z
CE001,2003-08-01,2003,8,EUROPE,heatwave,8,"European heatwave kills 70,000+",0,1,0,2026-06-05T18:27:37.561Z,2003-08-01,2026-06-05T18:46:32.255Z
CE005,2009-12-07,2009,12,GLOBAL,policy,8,Copenhagen Climate Conference COP15,1,0,0,2026-06-05T18:27:37.561Z,2009-12-07,2026-06-05T18:46:32.255Z
CE018,2018-11-08,2018,11,USA,wildfire,7,California Camp Fire (85 deaths),0,1,0,2026-06-05T18:27:37.561Z,2018-11-08,2026-06-05T18:46:32.255Z
CE047,2025-09-15,2025,9,ATLANTIC,hurricane,8,Multiple major Atlantic hurricanes,0,1,0,2026-06-05T18:27:37.561Z,2025-09-15,2026-06-05T18:46:32.255Z
CE028,2021-11-12,2021,11,GLOBAL,policy,8,COP26 Glasgow Climate Pact,1,0,0,2026-06-05T18:27:37.561Z,2021-11-12,2026-06-05T18:46:32.255Z
CE029,2022-03-15,2022,3,ANTARCTICA,heat,7,Antarctic +40°C anomaly,0,1,0,2026-06-05T18:27:37.561Z,2022-03-15,2026-06-05T18:46:32.255Z
CE033,2022-11-20,2022,11,GLOBAL,policy,8,COP27 Loss & Damage fund agreement,1,0,0,2026-06-05T18:27:37.561Z,2022-11-20,2026-06-05T18:46:32.255Z
